<table class="tfo-notebook-buttons" align="left">

  <td>
    <a target="_blank" href="https://colab.research.google.com/drive/1yrdikxLYAgJewvvsI_laTVIoomo-mmJP?usp=sharing"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Click here to run in Google Colab</a>
  </td>

</table>

#  A classification problem



In [ ]:
# Download data if it doesn't exist in the folder
!pip install wget
import wget
wget.download('https://raw.githubusercontent.com/NHERI-SimCenter/SimCenterAI_Workshop2021/master/notebooks/day1/Building_in_Hurricane.csv')

## 1. Step 1: Teaching - Load and review data

In [ ]:
# Read data into a pandas dataframe
import pandas as pd

all_data = pd.read_csv("Building_in_Hurricane.csv")

In [ ]:
# Print out a preview of the data table we loaded
# all_data

# Another option: print the first several rows
all_data.head(3)

In [ ]:
# List all the columns of the data table

all_data.columns.tolist()

In [ ]:
# Check types of values of a specific column

col = 'roof_shape'
series = all_data[col]
series_first_value = series[0]
print(series_first_value)
type(series[0])

- Notably, some columns contains numbers, which are likely to be ***"numerical values"***, such as "max_mph" with values like 5, 10, etc.
- On the contrary, some columns contains text, which are likely to be ***"categorical values"*** or ***"labels"***, such as "roof_shape" with values of "Complex".
- Different types of values need different strategy to review and process.

In [ ]:
# Check range of values for columns that contains numerical values
col = 'max_mph'
series = all_data[col]
print(series.min())
print(series.max())

# Generate a simple histogram to view the distribution clearly
series.hist()

In [ ]:
# Check unique values for columns that contains categorical values
col = 'roof_shape'
series = all_data[col]
print(series.unique())

In [ ]:
import matplotlib.pyplot as plt
X = series.unique()
Y = series.value_counts()
plt.bar(X, Y)
plt.show()

- Notably, even some columns contain numerical values, the values can be ***categorical***.
- For example, "overall_building_condition", although the values are 0, 1, 2, these are labels.

## 2. Data preprocessing
### Prepare input and output data

- Our task is to fit a model, which should consider the following columns as model input:

- **input** ['max_mph', 'age_yrs', 'number_of_stories',
       'roof_shape', 'roof_cover', 'wall_cladding',
       'structural_framing_system']

- The model should also consider the following column as the target of model outputs.

- **output** ['overall_building_condition']

In [ ]:
# Extract the columns for input

X = all_data.iloc[:, [1,2,3,4,5,6,7]].values
print('X shape: ', X.shape)
print('X: ', X)

In [ ]:
# Extract the columns for output target

Y = all_data.iloc[:, 0]
print('y shape: ', Y.shape)
print('Y: ', Y)

In [ ]:
# Review the output's unique values again

Y.value_counts()

In [ ]:
# Use a simple histogram to view the distribution of output values

Y.hist()

### Encode categorical variables

The model only reads numbers, not text. Therefore, for the columns with text-like categorical values, we should ***encode*** the values as number-like labels, such as 0, 1, 2, ...

In [ ]:
# Encoding Categorical Variables
from sklearn import preprocessing
le = preprocessing.LabelEncoder()

#roof_shape
X[:,3] = le.fit_transform(X[:,3])
mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(mapping)

In [ ]:
#roof_cover
X[:,4] = le.fit_transform(X[:,4])
mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(mapping)

In [ ]:
#wall_cladding
X[:,5] = le.fit_transform(X[:,5])
mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(mapping)

In [ ]:
#structural_framing_system
X[:,6] = le.fit_transform(X[:,6])
mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(mapping)

### Split into train and test sets

- To train machine learning models, especially neural networks, we should not only feed the data for ***training***, i.e., teaching the model the relationship between input and output, but also prepare the data for ***testing***, i.e., testing to what extend the model can predict the output correctly.
- The training and testing data should from the same distribution.
- In our case, we randomly split the data to two groups, one for training, one for testing.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state=1993)
# Here, test_size means the proportion of our data to be used as testing data, 0.2 means 20%.
# random_state is used to save how we split the data.
# In other words, if you did not change the value of random_state, the training and testing data will always be generated in the same way.

print('X_train shape: ', X_train.shape)
print('Y_train shape: ', Y_train.shape)
print('X_test shape: ', X_test.shape)
print('Y_test shape: ', Y_test.shape)


## 3. MODEL

## 3.1 Step 2: Learning - Multilayer Perceptron (Neural Network)

In [ ]:
from sklearn.neural_network import MLPClassifier

In [ ]:
# Build and train a neural network with 1 hidden layer of 10 perceptrons
model_1 = MLPClassifier(hidden_layer_sizes=(10,),max_iter=2000)
model_1.fit(X_train,Y_train)

In [ ]:
# Build and train a neural network with 2 hidden layers: the first layer has 20 perceptrons, the second has 30 perceptrons
model_2 = MLPClassifier(hidden_layer_sizes=(20, 30),max_iter=2000)
model_2.fit(X_train,Y_train)

## 3.2 Step 3: Inference - Validating the model with testing data

In [ ]:
# What will be the performance for Model 1, which has one layer?

from sklearn.metrics import f1_score

Y_pred = model_1.predict(X_test)
f1 = f1_score(Y_test.to_numpy(),Y_pred, average='micro')
print(f1)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Create confusion matrix
cm = confusion_matrix(Y_test, Y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# What will be the performance for Model 2, which has two layers?

from sklearn.metrics import f1_score

Y_pred = model_2.predict(X_test)
f1 = f1_score(Y_test.to_numpy(),Y_pred, average='micro')
print(f1)

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Create confusion matrix
cm = confusion_matrix(Y_test, Y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

## 3.2. Your assignment

Build and train 5 neural networks for the above data, with increasing layers and perceptrons.


Print the f1 score of them.


And, based on the f1 scores, can you make a conclusion:


Is it always good to increase the complexity of a model?

In [ ]:
# You code here:


You conclusion here (double click here to edit):




When you finish, don't forget to save this notebook. Then download it as a .ipynb and submit it.